In [1]:
import random

import carla
import cv2
import numpy as np

In [2]:
CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT_SECONDS = 10.0

IMAGE_WIDTH = 1280
IMAGE_HEIGHT = 720
CAMERA_FOV = 90.0

VEHICLE_FILTER = "vehicle.*"
RANDOM_SEED = 42

In [4]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(
        CARLA_HOST, 
        CARLA_PORT
    )

    client.set_timeout(
        CARLA_TIMEOUT_SECONDS
    )

    world = client.get_world()


    return client, world

In [5]:
def spawn_vehicle(
    world: carla.World,
    blueprint_library: carla.BlueprintLibrary,
) -> carla.Vehicle:

    blueprints = list(
        blueprint_library.filter(VEHICLE_FILTER)
    )

    if not blueprints:
        raise RuntimeError(
            "No vehicle blueprints available."
        )

    vehicle_blueprints = [
        blueprint
        for blueprint in blueprints
        if blueprint.has_attribute("number_of_wheels")
        and blueprint.get_attribute(
            "number_of_wheels"
        ).as_int() == 4
    ]

    vehicle_bp = random.choice(
        vehicle_blueprints or blueprints
    )

    spawn_points = list(
        world.get_map().get_spawn_points()
    )

    random.shuffle(spawn_points)

    for spawn_point in spawn_points:

        vehicle = world.try_spawn_actor(
            vehicle_bp,
            spawn_point,
        )

        if vehicle is not None:
            return vehicle

    raise RuntimeError(
        "Failed to spawn vehicle."
    )

In [36]:
def spawn_semantic_camera(
    world: carla.World,
    vehicle: carla.Vehicle,
) -> carla.Sensor:

    blueprint_library = (
        world.get_blueprint_library()
    )

    camera_bp = blueprint_library.find(
        "sensor.camera.semantic_segmentation"
    )

    camera_bp.set_attribute(
        "image_size_x",
        str(IMAGE_WIDTH),
    )

    camera_bp.set_attribute(
        "image_size_y",
        str(IMAGE_HEIGHT),
    )

    camera_bp.set_attribute(
        "fov",
        str(CAMERA_FOV),
    )

    camera_transform = carla.Transform(
        carla.Location(
            x=1.6,
            y=0.0,
            z=6.0
        )
    )

    camera = world.spawn_actor(
        camera_bp,
        camera_transform,
        attach_to=vehicle,
    )

    return camera

In [29]:
def sem_callback(image, data_dict):
    image.convert(carla.ColorConverter.CityScapesPalette)
    data_dict['sem_image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [32]:
def main() -> None:

    random.seed(RANDOM_SEED)

    client, world = connect_to_carla()
    blueprint_library = world.get_blueprint_library()

    vehicle = None
    camera = None

    data_dict = {
        "sem_image": None
    }

    try:

        # Spawn vehicle
        vehicle = spawn_vehicle(
            world,
            blueprint_library,
        )

        vehicle.set_autopilot(True)
        # Spawn semantic camera
        camera = spawn_semantic_camera(
            world,
            vehicle,
        )

        # Register callback
        camera.listen(
            lambda image: sem_callback(
                image,
                data_dict
            )
        )

        print("=== LIVE SEMANTIC SEGMENTATION ===")
        print("Press Q to exit.")

        # Live display loop
        while True:

            sem_image = data_dict["sem_image"]

            if sem_image is not None:
                cv2.imshow('All cameras', sem_image)

                # cv2.imshow(
                #     "CARLA - Semantic Segmentation",
                #     sem_image[:, :, :3]
                # )

            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break

    finally:

        if camera is not None and camera.is_alive:
            camera.stop()
            camera.destroy()

        if vehicle is not None and vehicle.is_alive:
            vehicle.destroy()

        cv2.destroyAllWindows()

        print("CARLA actors cleaned up.")

In [37]:
if __name__ == "__main__":
    main()

=== LIVE SEMANTIC SEGMENTATION ===
Press Q to exit.
CARLA actors cleaned up.
